# Age and online shopping frequency: gamma

**Goodman-Kruskal's Gamma** · a worked example from the StatNex teaching guide

**The story.** A retail association asked people in seven age groups how often they shop online: daily, weekly, monthly or rarely. 2,675 people answered. Both age group and shopping frequency are ordered, so we can ask about direction, not just association.

**The question.** As customers get older, do they shop online less often, and how strong is that trend?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/goodman_kruskal_gamma.csv` (8 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Title,Age group,Daily,Weekly,Monthly,Rarely,No answer,Total
Age,Age 19 or under,62.0,118.0,64.0,18.0,0.0,262.0
,Age 20-29,95.0,190.0,92.0,30.0,1.0,408.0
,Age 30-39,80.0,236.0,150.0,52.0,1.0,519.0
,Age 40-49,58.0,220.0,210.0,98.0,2.0,588.0
,Age 50-59,30.0,150.0,190.0,118.0,1.0,489.0
,Age 60-69,12.0,70.0,110.0,96.0,0.0,288.0
,Age 70 or over,4.0,22.0,40.0,60.0,1.0,127.0
,Total,341.0,1006.0,856.0,472.0,6.0,2681.0

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# A table of counts whose rows and columns are both ordered: the rows in the order of
# the row variable, the columns in the order of the column variable.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Daily", "Weekly", "Monthly", "Rarely"]
table = data.set_index("Age group")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
from totals import is_total

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank count is left out.
table = table.dropna()
counts = table.to_numpy(float)

## Observed frequency

In [ ]:
observed = pd.concat([table, table.sum(axis=1).rename("Total")], axis=1)
observed = pd.concat([observed, observed.sum().to_frame("Total").T])
print(observed)

## Worksheet: concordant and discordant partners

In [ ]:
# Every pair of people is compared on both variables. A pair is concordant when the
# person further down the table (older, say) is also further right (shops less often),
# and discordant when they are further left. For each cell, its concordant partners are
# the cases strictly below and to the right, its discordant partners those strictly
# below and to the left; P and Q add up count x partners over the cells. Pairs in the
# same row or the same column are ties and count in neither.
rows = []
for i, row in enumerate(table.index):
    for j, column in enumerate(table.columns):
        concordant = counts[i + 1:, j + 1:].sum()
        discordant = counts[i + 1:, :j].sum()
        rows.append([row, column, counts[i, j], concordant, counts[i, j] * concordant,
                     discordant, counts[i, j] * discordant])
worksheet = pd.DataFrame(rows, columns=["row", "column", "count", "concordant", "P part",
                                        "discordant", "Q part"])
print()
print(worksheet.to_string(index=False))

## Measures of ordinal association

In [ ]:
# All three are (P - Q) over a different denominator, and run from -1 to +1; a positive
# value means the two orderings agree (further down the table goes with further right).
# Gamma = (P - Q) / (P + Q) ignores ties, so it is always the largest. Kendall's tau-b
# also counts the pairs tied on one variable, and reaches +-1 only on a square table;
# tau-c rescales for a rectangular one (m = the smaller of the number of rows and
# columns). SciPy has no gamma, so it is computed from P and Q; tau-b and tau-c come from
# scipy.stats.kendalltau run on one row per case, rebuilt from the counts.
P, Q = worksheet["P part"].sum(), worksheet["Q part"].sum()
gamma = (P - Q) / (P + Q)
row_of_case = np.repeat(np.repeat(np.arange(counts.shape[0]), counts.shape[1]), counts.ravel().astype(int))
column_of_case = np.repeat(np.tile(np.arange(counts.shape[1]), counts.shape[0]), counts.ravel().astype(int))
tau_b = stats.kendalltau(row_of_case, column_of_case, variant="b").statistic
tau_c = stats.kendalltau(row_of_case, column_of_case, variant="c").statistic
measures = pd.Series({"Goodman-Kruskal's gamma": gamma, "Kendall's tau-b": tau_b,
                      "Kendall's tau-c": tau_c})
print()
print(f"P = {P:.0f} concordant pairs, Q = {Q:.0f} discordant pairs")
print(measures)

## What the result says

Gamma compares pairs of people who differ in both age group and shopping frequency. A pair is concordant when the older person is also further along the columns, which run from Daily to Rarely. It is discordant when the order is reversed. Gamma is the balance of the two: 0 means no trend and 1 means a perfect one.

For the 2,675 people, gamma is 0.37. It is positive, so concordant pairs are in the majority: the older person is usually the one who shops less often. Kendall's tau-b (0.29) and tau-c (0.30) also count tied pairs, so they are a little smaller.

So yes, older customers shop online less often, and the trend is moderate, not strong.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.